# Pre-application distribution × permit volume comparison

Six scenarios: `lognormal_10`, `lognormal_60`, and `lognormal_180` pre-application distributions, each run with **2,000** and **6,500** permits.

**Outputs**
- Summary table: mean **disaster → ready for construction** and **plan application → ready for construction** (pooled across permits, plus run-level uncertainty).
- **2×3 grid**: average waiting vs service time by process step (permits pooled across Monte Carlo runs per scenario).
- **2×3 grid**: mean **implied** staff utilization over time (active + waiting demand / capacity) (average across runs).

Reduce `N_RUNS` for a quick smoke test; increase for smoother Monte Carlo means.



In [1]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import run_simulation
importlib.reload(run_simulation)
from run_simulation import run_multiple_simulations, plot_staff_utilization_series

import visualize_permits
importlib.reload(visualize_permits)
from visualize_permits import plot_average_waiting_and_service_by_step

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

# --- Monte Carlo & shared staffing (match your main baseline unless you edit) ---
N_RUNS = 100
BASE_SEED = 42
SEQUENTIAL_MODE = "standard"
PERMIT_MIX = "balanced"
SIMULATION_DURATION = None
UTILIZATION_STEP = 0.05
UTILIZATION_KIND = "actual"  # "actual" or "implied"

PRE_APPLICATION_DISTS = ["lognormal_10", "lognormal_60", "lognormal_180"]
PERMIT_COUNTS = [2000, 6500]

SHARED_SCENARIO_KNOBS = {
    "sequential": SEQUENTIAL_MODE,
    "ai_review": "none",
    "permit_mix": PERMIT_MIX,
    "review_duration_families": None,
    "review_duration_multipliers": None,
    "planning_staff_count": 20,
    "planning_caseload_per_staff": 7,
    "public_works_staff_count": 30,
    "public_works_caseload_per_staff": 7,
    "fire_staff_count": 10,
    "fire_caseload_per_staff": 7,
}


def scenario_key(dist: str, n: int) -> str:
    return f"{dist} | n={n:,}"


def row_col_for(n_permits: int, dist: str):
    row = PERMIT_COUNTS.index(n_permits)
    col = PRE_APPLICATION_DISTS.index(dist)
    return row, col


# Explicit scenario order: rows = permit count, cols = distribution
SCENARIO_PARAMS_LIST = []
for n in PERMIT_COUNTS:
    for dist in PRE_APPLICATION_DISTS:
        SCENARIO_PARAMS_LIST.append(
            {
                "name": scenario_key(dist, n),
                "num_permits": n,
                "pre_application_distribution": dist,
                **SHARED_SCENARIO_KNOBS,
            }
        )

n_tasks = len(SCENARIO_PARAMS_LIST) * N_RUNS
print(f"Total simulation runs: {n_tasks} ({len(SCENARIO_PARAMS_LIST)} scenarios × {N_RUNS} runs)")



Total simulation runs: 600 (6 scenarios × 100 runs)


In [2]:
results, average_staff_util_by_scenario = run_multiple_simulations(
    n_runs=N_RUNS,
    num_permits=6500,
    simulation_duration=SIMULATION_DURATION,
    base_seed=BASE_SEED,
    scenario_params_list=SCENARIO_PARAMS_LIST,
    collect_permits=True,
    collect_average_staff_utilization=True,
    utilization_step=UTILIZATION_STEP,
    utilization_kind=UTILIZATION_KIND,
)

permits_by_scenario = {}
for entry in results:
    permits_by_scenario.setdefault(entry["scenario"], []).extend(entry.get("permits", []))

print("Completed. Permits collected per scenario:")
for scenario_def in SCENARIO_PARAMS_LIST:
    name = scenario_def["name"]
    n_expect = scenario_def["num_permits"]
    plist = permits_by_scenario.get(name, [])
    print(f"  {name}: {len(plist):,} permits (expected {N_RUNS * n_expect:,})")



KeyboardInterrupt: 

In [ ]:
def disaster_to_ready_days(p):
    if p.ready_for_construction is None or p.created_at is None:
        return None
    return float(p.ready_for_construction - p.created_at)


def application_to_ready_days(p):
    if p.ready_for_construction is None or p.planning_request is None:
        return None
    return float(p.ready_for_construction - p.planning_request)


rows = []
for scenario_def in SCENARIO_PARAMS_LIST:
    scenario_name = scenario_def["name"]
    perms = permits_by_scenario.get(scenario_name, [])
    dvals = [disaster_to_ready_days(p) for p in perms]
    avals = [application_to_ready_days(p) for p in perms]
    dvals = [x for x in dvals if x is not None]
    avals = [x for x in avals if x is not None]

    run_disaster_means = []
    run_app_means = []
    for entry in results:
        if entry["scenario"] != scenario_name:
            continue
        rp = entry.get("permits", [])
        rd = [disaster_to_ready_days(p) for p in rp]
        ra = [application_to_ready_days(p) for p in rp]
        rd = [x for x in rd if x is not None]
        ra = [x for x in ra if x is not None]
        if rd:
            run_disaster_means.append(float(np.mean(rd)))
        if ra:
            run_app_means.append(float(np.mean(ra)))

    rows.append(
        {
            "pre_application_distribution": scenario_def["pre_application_distribution"],
            "num_permits": scenario_def["num_permits"],
            "pooled_mean_disaster_to_ready_days": float(np.mean(dvals)) if dvals else np.nan,
            "pooled_mean_application_to_ready_days": float(np.mean(avals)) if avals else np.nan,
            "run_mean_disaster_to_ready_mean": float(np.mean(run_disaster_means)) if run_disaster_means else np.nan,
            "run_mean_disaster_to_ready_std": float(np.std(run_disaster_means, ddof=1)) if len(run_disaster_means) > 1 else 0.0,
            "run_mean_application_to_ready_mean": float(np.mean(run_app_means)) if run_app_means else np.nan,
            "run_mean_application_to_ready_std": float(np.std(run_app_means, ddof=1)) if len(run_app_means) > 1 else 0.0,
            "n_permits_pooled": len(dvals),
        }
    )

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.sort_values(["num_permits", "pre_application_distribution"]).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
metrics_df


,pre_application_distribution,num_permits,pooled_mean_disaster_to_ready_days,pooled_mean_application_to_ready_days,run_mean_disaster_to_ready_mean,run_mean_disaster_to_ready_std,run_mean_application_to_ready_mean,run_mean_application_to_ready_std,n_permits_pooled
0,lognormal_10,2000,172.710248,164.039788,172.710248,1.787830,164.039788,1.779868,200000
1,lognormal_180,2000,274.091954,118.023673,274.091954,3.243916,118.023673,1.292007,200000
2,lognormal_60,2000,180.226048,128.203288,180.226048,1.618012,128.203288,1.740167,200000
3,lognormal_10,6500,324.064737,315.415473,324.064737,2.104035,315.415473,2.116037,650000
4,lognormal_180,6500,363.202599,207.515852,363.202599,3.207467,207.515852,3.272313,650000
5,lognormal_60,6500,318.828648,266.933066,318.828648,1.623334,266.933066,1.664902,650000


In [ ]:
# Optional: save metrics for thesis tables
# metrics_df.to_csv("preapp_volume_scenario_metrics.csv", index=False)

import importlib

import run_simulation

importlib.reload(run_simulation)

from run_simulation import plot_staff_utilization_series

import visualize_permits

importlib.reload(visualize_permits)

from visualize_permits import plot_average_waiting_and_service_by_step

label_map = {
    "EPA Debris": "EPA debris",
    "USACE Debris": "USACE debris",
    "Pre-Application Activities": "Pre-application",
    "Planning": "Planning",
    "Special Zoning": "Special zoning",
    "Public Works": "Public works",
    "Agency Referral": "Agency referral",
    "Fire Review": "Fire review",
    "Applicant Revisions": "Applicant revisions",
}

# Thesis-scale typography (matplotlib pt)
FIG_SUPTITLE = 22
PANEL_TITLE = 20
PANEL_AXIS = 18
PANEL_TICK = 16
PANEL_LEGEND = 16
BAR_VALUES = 10
UTIL_LINE_WIDTH = 3.0
FIG_LEGEND = 18

# Easy title controls
WAITING_FIGURE_TITLES = {
    2000: "Average waiting vs service time by process step (2,000 permits)",
    6500: "Average waiting vs service time by process step (6,500 permits)",
}
UTILIZATION_FIGURE_TITLES = {
    2000: "Mean staff utilization over time (2,000 permits)",
    6500: "Mean staff utilization over time (6,500 permits)",
}
PANEL_TITLES = {
    "lognormal_10": "\u03bc = 10",
    "lognormal_60": "\u03bc = 60",
    "lognormal_180": "\u03bc = 180",
}

# Extra headroom for waiting/service bars by permit count.
# Increase 6,500 to ensure all bars/labels fit.
WAITING_Y_PAD_BY_PERMITS = {
    2000: 1.05,
    6500: 1.30,
}

for n in PERMIT_COUNTS:
    fig_w, axes_w = plt.subplots(1, 3, figsize=(22, 6.8), sharey=True)
    fig_w.suptitle(
        WAITING_FIGURE_TITLES.get(n, f"Average waiting vs service time by process step ({n:,} permits; pooled across runs)"),
        fontsize=FIG_SUPTITLE,
        fontweight="bold",
        y=1.02,
    )

    for col, dist in enumerate(PRE_APPLICATION_DISTS):
        sname = scenario_key(dist, n)
        ax = axes_w[col]
        plist = permits_by_scenario.get(sname, [])
        plot_average_waiting_and_service_by_step(
            plist,
            label_map=label_map,
            ax=ax,
            title=PANEL_TITLES.get(dist, dist),
            silent=True,
            title_size=PANEL_TITLE,
            axis_label_size=PANEL_AXIS,
            tick_label_size=PANEL_TICK,
            legend_size=PANEL_LEGEND,
            bar_value_size=BAR_VALUES,
        )
        if col > 0:
            ax.set_ylabel("")

    current_ymax = max(ax.get_ylim()[1] for ax in axes_w)
    padded_ymax = current_ymax * WAITING_Y_PAD_BY_PERMITS.get(n, 1.05)
    for ax in axes_w:
        ax.set_ylim(0, padded_ymax)

    plt.tight_layout()
    plt.show()

for n in PERMIT_COUNTS:
    fig_u, axes_u = plt.subplots(1, 3, figsize=(22, 6.8), sharex=True, sharey=True)
    fig_u.suptitle(
        UTILIZATION_FIGURE_TITLES.get(n, f"Mean implied staff utilization over time ({n:,} permits; average across Monte Carlo runs)"),
        fontsize=FIG_SUPTITLE,
        fontweight="bold",
        y=1.02,
    )

    for col, dist in enumerate(PRE_APPLICATION_DISTS):
        sname = scenario_key(dist, n)
        ax = axes_u[col]
        util = average_staff_util_by_scenario[sname]
        plot_staff_utilization_series(
            util,
            title=PANEL_TITLES.get(dist, dist),
            ax=ax,
            legend=False,
            xlim=(0, 25),
            title_size=PANEL_TITLE,
            axis_label_size=PANEL_AXIS,
            tick_label_size=PANEL_TICK,
            line_width=UTIL_LINE_WIDTH,
        )
        if col > 0:
            ax.set_ylabel("")

    handles, labels = axes_u[0].get_legend_handles_labels()
    fig_u.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        bbox_to_anchor=(0.5, 0.02),
        fontsize=FIG_LEGEND,
    )

    plt.tight_layout(rect=(0, 0.06, 1, 0.98))
    plt.show()




NameError: name 'PERMIT_COUNTS' is not defined

In [ ]:
# Focused staffing comparison: lognormal_180 with 2,000 permits
FOCUS_PREAPP = "lognormal_180"
FOCUS_PERMITS = 2000

STAFFING_SCENARIOS = {
    "low": {"planning_staff_count": 2, "public_works_staff_count": 3, "fire_staff_count": 1},
    "medium": {"planning_staff_count": 8, "public_works_staff_count": 12, "fire_staff_count": 4},
    "high": {"planning_staff_count": 20, "public_works_staff_count": 30, "fire_staff_count": 10},
}

FOCUS_SCENARIO_PARAMS = []
for staffing_name, staffing in STAFFING_SCENARIOS.items():
    FOCUS_SCENARIO_PARAMS.append(
        {
            "name": f"{FOCUS_PREAPP} | n={FOCUS_PERMITS:,} | staffing={staffing_name}",
            "num_permits": FOCUS_PERMITS,
            "pre_application_distribution": FOCUS_PREAPP,
            **SHARED_SCENARIO_KNOBS,
            **staffing,
        }
    )

focus_results, focus_average_staff_util = run_multiple_simulations(
    n_runs=N_RUNS,
    num_permits=FOCUS_PERMITS,
    simulation_duration=SIMULATION_DURATION,
    base_seed=BASE_SEED,
    scenario_params_list=FOCUS_SCENARIO_PARAMS,
    collect_permits=True,
    collect_average_staff_utilization=True,
    utilization_step=UTILIZATION_STEP,
    utilization_kind=UTILIZATION_KIND,
)

focus_permits_by_scenario = {}
for entry in focus_results:
    focus_permits_by_scenario.setdefault(entry["scenario"], []).extend(entry.get("permits", []))


NameError: name 'SHARED_SCENARIO_KNOBS' is not defined

In [ ]:
# Panel 1: waiting vs service by staffing level
fig_ws, axes_ws = plt.subplots(1, 3, figsize=(22, 6.8), sharey=True)
fig_ws.suptitle(
    "Average waiting vs service time by process step (lognormal_180, 2,000 permits)",
    fontsize=FIG_SUPTITLE,
    fontweight="bold",
    y=1.02,
)

for col, staffing_name in enumerate(["low", "medium", "high"]):
    scenario_name = f"{FOCUS_PREAPP} | n={FOCUS_PERMITS:,} | staffing={staffing_name}"
    ax = axes_ws[col]
    plot_average_waiting_and_service_by_step(
        focus_permits_by_scenario.get(scenario_name, []),
        label_map=label_map,
        ax=ax,
        title=staffing_name.capitalize(),
        silent=True,
        title_size=PANEL_TITLE,
        axis_label_size=PANEL_AXIS,
        tick_label_size=PANEL_TICK,
        legend_size=PANEL_LEGEND,
        bar_value_size=BAR_VALUES,
    )
    if col > 0:
        ax.set_ylabel("")

# Standard y-axis range that includes the tallest bar.
all_bar_heights = [patch.get_height() for ax in axes_ws for patch in ax.patches]
y_top = (max(all_bar_heights) * 1.05) if all_bar_heights else 1.0
for ax in axes_ws:
    ax.set_ylim(0, y_top)

plt.tight_layout()
plt.show()

# Panel 2: corresponding staff utilization by staffing level
fig_util, axes_util = plt.subplots(1, 3, figsize=(22, 6.8), sharex=True, sharey=True)
fig_util.suptitle(
    "Mean staff utilization over time (\u03bc = 180, 2,000 permits)",
    fontsize=FIG_SUPTITLE,
    fontweight="bold",
    y=1.02,
)

for col, staffing_name in enumerate(["low", "medium", "high"]):
    scenario_name = f"{FOCUS_PREAPP} | n={FOCUS_PERMITS:,} | staffing={staffing_name}"
    ax = axes_util[col]
    plot_staff_utilization_series(
        focus_average_staff_util[scenario_name],
        title=staffing_name.capitalize(),
        ax=ax,
        legend=False,
        xlim=(0, 50),
        title_size=PANEL_TITLE,
        axis_label_size=PANEL_AXIS,
        tick_label_size=PANEL_TICK,
        line_width=UTIL_LINE_WIDTH,
    )
    if col > 0:
        ax.set_ylabel("")

handles, labels = axes_util[0].get_legend_handles_labels()
fig_util.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0.02),
    fontsize=FIG_LEGEND,
)

plt.tight_layout(rect=(0, 0.06, 1, 0.98))
plt.show()


NameError: name 'plt' is not defined